# 06a — Scenario B Candidate Build (Slice → Candidates → Artifacts)

This notebook creates **Scenario B** from our clinical trials metadata layer by:

1) Loading trial metadata (prefer full parquet; fall back to sample if needed)  
2) Defining Scenario B as a reproducible **slice rule** (default: top sponsors)  
3) Building a compact **candidates table** with consistent scoring columns:
   - `_benefit_raw`, `_cost_raw`, `_safety_raw`
4) Writing artifacts used by downstream optimization:
   - `data/scenarios/scenario_B_trial_ids.csv`
   - `data/processed/scenario_B_candidates.csv`

This keeps the “big data” piece upstream (S3 corpus → metadata) but produces a small, auditable optimization instance downstream.


In [1]:
# ============================================================
# Cell 1 — Setup: imports, paths, and output directories
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# --- Project directories ---
DATA_DIR = Path("data")
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
SCENARIO_DIR = DATA_DIR / "scenarios"
RESULTS_DIR = DATA_DIR / "results"

for d in [INTERIM_DIR, PROCESSED_DIR, SCENARIO_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Preferred full metadata (local-only), plus git-safe fallback samples ---
PATH_META_FULL = INTERIM_DIR / "clinical_trials_metadata.parquet"
PATH_META_SAMPLE_PARQUET = INTERIM_DIR / "clinical_trials_metadata_sample.parquet"
PATH_META_SAMPLE_CSV = INTERIM_DIR / "clinical_trials_metadata_sample.csv"

# --- Outputs ---
PATH_SCENARIO_B_IDS = SCENARIO_DIR / "scenario_B_trial_ids.csv"
PATH_SCENARIO_B_CANDIDATES = PROCESSED_DIR / "scenario_B_candidates.csv"

print("OK: Directories ready.")
print("Full metadata:", PATH_META_FULL)
print("Sample parquet:", PATH_META_SAMPLE_PARQUET)
print("Sample csv:", PATH_META_SAMPLE_CSV)
print("Output IDs:", PATH_SCENARIO_B_IDS)
print("Output candidates:", PATH_SCENARIO_B_CANDIDATES)


OK: Directories ready.
Full metadata: data/interim/clinical_trials_metadata.parquet
Sample parquet: data/interim/clinical_trials_metadata_sample.parquet
Sample csv: data/interim/clinical_trials_metadata_sample.csv
Output IDs: data/scenarios/scenario_B_trial_ids.csv
Output candidates: data/processed/scenario_B_candidates.csv


### What Cell 1 Just Did

- Established standard project paths and output locations for Scenario B artifacts.
- Declared a “prefer full / fall back to sample” loading strategy so the notebook runs in multiple environments.


In [2]:
# ============================================================
# Cell 2 — Load metadata (full parquet preferred; sample fallback)
# ============================================================

def load_trials_metadata():
    if PATH_META_FULL.exists():
        df = pd.read_parquet(PATH_META_FULL)
        source = str(PATH_META_FULL)
        return df, source

    if PATH_META_SAMPLE_PARQUET.exists():
        df = pd.read_parquet(PATH_META_SAMPLE_PARQUET)
        source = str(PATH_META_SAMPLE_PARQUET)
        return df, source

    if PATH_META_SAMPLE_CSV.exists():
        df = pd.read_csv(PATH_META_SAMPLE_CSV)
        source = str(PATH_META_SAMPLE_CSV)
        return df, source

    raise FileNotFoundError(
        "Could not find trials metadata.\n"
        f"Tried:\n  - {PATH_META_FULL}\n  - {PATH_META_SAMPLE_PARQUET}\n  - {PATH_META_SAMPLE_CSV}\n\n"
        "Fix: run Notebook 01 ingestion to create the metadata parquet (local), or ensure sample artifacts exist."
    )

trials_meta, meta_source = load_trials_metadata()

print("Loaded trials metadata from:", meta_source)
print("Shape:", trials_meta.shape)
print("Columns (first 30):", list(trials_meta.columns)[:30])

# Minimal required identifier check
id_candidates = ["nct_id", "NCTId", "nctid", "trial_id"]
NCT_COL = next((c for c in id_candidates if c in trials_meta.columns), None)
if NCT_COL is None:
    raise ValueError(f"Could not find an NCT id column among {id_candidates}. Found: {list(trials_meta.columns)}")

# Normalize to 'nct_id'
if NCT_COL != "nct_id":
    trials_meta = trials_meta.rename(columns={NCT_COL: "nct_id"})

trials_meta["nct_id"] = trials_meta["nct_id"].astype(str)
print("OK: Using id column: nct_id")


Loaded trials metadata from: data/interim/clinical_trials_metadata.parquet
Shape: (557292, 14)
Columns (first 30): ['nct_id', 'brief_title', 'official_title', 'overall_status', 'phase', 'conditions', 'interventions', 'enrollment', 'location_countries', 'lead_sponsor', 'size_bytes', 'last_modified', 'folder', 's3_key']
OK: Using id column: nct_id


### What Cell 2 Just Did

- Loaded the trial metadata layer (full parquet if present; otherwise sample artifacts).
- Normalized the trial identifier to `nct_id` so Scenario B downstream files are consistent.


In [3]:
# ============================================================
# Cell 3 — Define Scenario B slice (default: Top N sponsors) + build candidate pool
# ============================================================

# --- Scenario B definition knobs (edit here, run-all below) ---
TOP_N_SPONSORS = 25          # Scenario B concept: "Trials from top N sponsors"
MAX_CANDIDATES = 60          # keep optimization instance compact
STATUS_ALLOW = {"RECRUITING", "ACTIVE_NOT_RECRUITING", "ENROLLING_BY_INVITATION"}
PHASE_ALLOW = {"PHASE2", "PHASE3", "PHASE2_PHASE3"}  # relaxed; adjust as needed

# --- Column helpers ---
def first_present(candidates):
    return next((c for c in candidates if c in trials_meta.columns), None)

SPONSOR_COL = first_present(["lead_sponsor_norm", "lead_sponsor", "sponsor", "sponsor_name"])
STATUS_COL  = first_present(["overall_status", "status"])
PHASE_COL   = first_present(["phase", "phase_simplified"])
REGION_COL  = first_present(["region_label", "location_region", "region"])

if SPONSOR_COL is None:
    raise ValueError("No sponsor column found (lead_sponsor_norm/lead_sponsor/sponsor).")
if STATUS_COL is None:
    raise ValueError("No status column found (overall_status/status).")
if PHASE_COL is None:
    raise ValueError("No phase column found (phase/phase_simplified).")

df = trials_meta.copy()

# Normalize strings for filtering
df[STATUS_COL] = df[STATUS_COL].astype(str).str.upper().str.strip()
df[PHASE_COL]  = df[PHASE_COL].astype(str).str.upper().str.replace(" ", "").str.strip()
df[SPONSOR_COL] = df[SPONSOR_COL].astype(str).str.strip()

# Filter by status/phase
df = df[df[STATUS_COL].isin(STATUS_ALLOW)]
df = df[df[PHASE_COL].isin(PHASE_ALLOW)]

# Top sponsors by count
top_sponsors = (
    df[SPONSOR_COL]
    .value_counts(dropna=True)
    .head(TOP_N_SPONSORS)
    .index
    .tolist()
)

df = df[df[SPONSOR_COL].isin(top_sponsors)].copy()

print("Scenario B slice:")
print("  Status allow:", sorted(STATUS_ALLOW))
print("  Phase allow:", sorted(PHASE_ALLOW))
print("  Top sponsors selected:", len(top_sponsors))
print("  Post-slice rows:", len(df))

# ---- Build consistent scoring components ----
# We'll create _benefit_raw, _cost_raw, _safety_raw using best-available columns.
BENEFIT_COL = first_present(["benefit_score", "benefit", "benefit_raw", "enrollment_feasibility_score"])
COST_COL    = first_present(["estimated_trial_cost", "cost", "trial_cost", "cost_estimate"])
SAFETY_COL  = first_present(["sponsor_safety_score", "safety_score", "safety_risk_score", "risk_score"])

# If missing, create reasonable placeholders (still auditable and deterministic)
if BENEFIT_COL is None:
    # fallback: prefer later phase as higher benefit proxy
    phase_map = {"PHASE2": 0.6, "PHASE2_PHASE3": 0.8, "PHASE3": 1.0}
    df["_benefit_raw"] = df[PHASE_COL].map(phase_map).fillna(0.5).astype(float)
    benefit_note = "proxy_from_phase"
else:
    df["_benefit_raw"] = pd.to_numeric(df[BENEFIT_COL], errors="coerce").fillna(0.0).astype(float)
    benefit_note = f"from:{BENEFIT_COL}"

if COST_COL is None:
    # fallback: approximate cost proxy from phase
    cost_map = {"PHASE2": 1.0, "PHASE2_PHASE3": 1.5, "PHASE3": 2.0}
    df["_cost_raw"] = df[PHASE_COL].map(cost_map).fillna(1.0).astype(float)
    cost_note = "proxy_from_phase"
else:
    df["_cost_raw"] = pd.to_numeric(df[COST_COL], errors="coerce").fillna(df[COST_COL].median() if COST_COL in df else 0.0).astype(float)
    cost_note = f"from:{COST_COL}"

if SAFETY_COL is None:
    # fallback: neutral safety (all equal) — keeps pipeline running but flags missing feature
    df["_safety_raw"] = 0.0
    safety_note = "missing_set_to_0"
else:
    df["_safety_raw"] = pd.to_numeric(df[SAFETY_COL], errors="coerce").fillna(0.0).astype(float)
    safety_note = f"from:{SAFETY_COL}"

print("\nScoring component sources:")
print("  benefit:", benefit_note)
print("  cost   :", cost_note)
print("  safety :", safety_note)

# ---- Candidate selection (keep compact for optimization) ----
# Default: prioritize higher benefit, lower cost, lower safety risk.
# We turn it into a selection score to choose MAX_CANDIDATES.
# (Downstream QUBO will use separate lambdas; here we just choose a manageable pool.)
safe_z = (df["_safety_raw"] - df["_safety_raw"].mean()) / (df["_safety_raw"].std() + 1e-9)
cost_z = (df["_cost_raw"] - df["_cost_raw"].mean()) / (df["_cost_raw"].std() + 1e-9)
ben_z  = (df["_benefit_raw"] - df["_benefit_raw"].mean()) / (df["_benefit_raw"].std() + 1e-9)

df["_pool_score"] = (ben_z) - 0.5 * cost_z - 0.5 * safe_z

# Keep only relevant columns for candidates artifact
keep_cols = ["nct_id", SPONSOR_COL, STATUS_COL, PHASE_COL]
if REGION_COL is not None:
    keep_cols.append(REGION_COL)

keep_cols += ["_benefit_raw", "_cost_raw", "_safety_raw", "_pool_score"]

# If titles exist, include them for readability
TITLE_COL = first_present(["brief_title", "official_title", "title"])
if TITLE_COL is not None:
    keep_cols.insert(1, TITLE_COL)

cand = (
    df[keep_cols]
    .drop_duplicates(subset=["nct_id"])
    .sort_values("_pool_score", ascending=False)
    .head(MAX_CANDIDATES)
    .reset_index(drop=True)
)

print("\nScenario B candidates:", cand.shape)
display(cand.head(10))


Scenario B slice:
  Status allow: ['ACTIVE_NOT_RECRUITING', 'ENROLLING_BY_INVITATION', 'RECRUITING']
  Phase allow: ['PHASE2', 'PHASE2_PHASE3', 'PHASE3']
  Top sponsors selected: 25
  Post-slice rows: 1971

Scoring component sources:
  benefit: proxy_from_phase
  cost   : proxy_from_phase
  safety : missing_set_to_0

Scenario B candidates: (60, 9)


,nct_id,brief_title,lead_sponsor,overall_status,phase,_benefit_raw,_cost_raw,_safety_raw,_pool_score
0,NCT06760637,Study of PF-07220060 With Letrozole in Adults ...,Pfizer,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
1,NCT06713616,PCORI Comparative Effectiveness Study-Esketami...,Yale University,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
2,NCT06711887,Phase III Extension Study of Efficacy and Safe...,Novartis Pharmaceuticals,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
3,NCT06706817,A Study to Investigate Changes in Symptoms in ...,AstraZeneca,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
4,NCT05674305,Radiotherapy Alone Versus Concurrent Chemo-rad...,Fudan University,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
5,NCT06703476,A Study of Surgical Techniques During Cystectomy,Memorial Sloan Kettering Cancer Center,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
6,NCT05675410,A Study to Compare Standard Therapy to Treat H...,National Cancer Institute (NCI),RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
7,NCT06701331,Safety and Efficacy of Upadacitinib in Combina...,AbbVie,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
8,NCT05677451,24 Weeks Double-blind Randomized Placebo-contr...,Novartis Pharmaceuticals,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
9,NCT06698796,A Study to Understand How the Study Medicine D...,Pfizer,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893


### What Cell 3 Just Did

- Defined **Scenario B** as a reproducible slice (default: Trials from the top N sponsors, bounded by status and phase).
- Built a compact candidate pool suitable for QUBO/QAOA by:
  - constructing consistent scoring components (`_benefit_raw`, `_cost_raw`, `_safety_raw`) using best-available columns,
  - and selecting the top candidates by a transparent pool score.
- This cell’s output (`cand`) is the single source of truth for Scenario B optimization.


In [5]:
# ============================================================
# Cell 4 — Write Scenario B artifacts (IDs + candidates)
# ============================================================

# Write trial IDs
cand[["nct_id"]].to_csv(PATH_SCENARIO_B_IDS, index=False)

# Write candidates table
cand.to_csv(PATH_SCENARIO_B_CANDIDATES, index=False)

print("Wrote:")
print("  -", PATH_SCENARIO_B_IDS)
print("  -", PATH_SCENARIO_B_CANDIDATES)

print("\nCounts by sponsor (top 10):")
display(cand.groupby(cand.columns[cand.columns.isin([SPONSOR_COL])][0])["nct_id"].count().sort_values(ascending=False).head(10))


Wrote:
  - data/scenarios/scenario_B_trial_ids.csv
  - data/processed/scenario_B_candidates.csv

Counts by sponsor (top 10):


lead_sponsor
Novartis Pharmaceuticals                   8
Sun Yat-sen University                     7
Pfizer                                     6
Assistance Publique - Hôpitaux de Paris    5
AstraZeneca                                5
AbbVie                                     4
Merck Sharp & Dohme LLC                    3
Fudan University                           3
Eli Lilly and Company                      3
National Cancer Institute (NCI)            2
Name: nct_id, dtype: int64

### What Cell 4 Just Did

- Wrote the two canonical Scenario B artifacts used by downstream notebooks:
  - `data/scenarios/scenario_B_trial_ids.csv` (the slice ID list)
  - `data/processed/scenario_B_candidates.csv` (the scored pool used to build QUBO)
- These files are the “handoff point” from data engineering to optimization.


In [6]:
# ============================================================
# Cell 5 — Sanity checks on Scenario B candidates (quality gates)
# ============================================================

required = ["nct_id", "_benefit_raw", "_cost_raw", "_safety_raw"]
missing = [c for c in required if c not in cand.columns]
if missing:
    raise ValueError(f"Scenario B candidates missing required columns: {missing}")

if cand["nct_id"].isna().any():
    raise ValueError("Found NaN nct_id in Scenario B candidates.")

dup_n = cand["nct_id"].duplicated().sum()
if dup_n > 0:
    raise ValueError(f"Found duplicate nct_id rows in candidates: {dup_n}")

print("OK: Candidate quality gates passed.")
print("Candidates:", len(cand))
print("Benefit range:", float(cand["_benefit_raw"].min()), "to", float(cand["_benefit_raw"].max()))
print("Cost range   :", float(cand["_cost_raw"].min()), "to", float(cand["_cost_raw"].max()))
print("Safety range :", float(cand["_safety_raw"].min()), "to", float(cand["_safety_raw"].max()))


OK: Candidate quality gates passed.
Candidates: 60
Benefit range: 1.0 to 1.0
Cost range   : 2.0 to 2.0
Safety range : 0.0 to 0.0


### What Cell 5 Just Did

- Enforced minimal quality gates on Scenario B candidates (required columns, no missing IDs, no duplicates).
- Printed quick ranges for benefit/cost/safety so we can spot obviously broken feature generation early.


In [7]:
# ============================================================
# Cell 6 — Save a tiny “Scenario B manifest” (provenance snapshot)
# ============================================================

import json
from datetime import datetime

manifest = {
    "scenario": "B",
    "created_at_local": datetime.now().isoformat(timespec="seconds"),
    "metadata_source": meta_source,
    "top_n_sponsors": int(TOP_N_SPONSORS),
    "max_candidates": int(MAX_CANDIDATES),
    "status_allow": sorted(list(STATUS_ALLOW)),
    "phase_allow": sorted(list(PHASE_ALLOW)),
    "columns_used": {
        "sponsor_col": SPONSOR_COL,
        "status_col": STATUS_COL,
        "phase_col": PHASE_COL,
        "region_col": REGION_COL,
        "title_col": TITLE_COL,
        "benefit_source": benefit_note,
        "cost_source": cost_note,
        "safety_source": safety_note
    },
    "outputs": {
        "scenario_ids_csv": str(PATH_SCENARIO_B_IDS),
        "scenario_candidates_csv": str(PATH_SCENARIO_B_CANDIDATES)
    }
}

PATH_SCENARIO_B_MANIFEST = RESULTS_DIR / "scenario_B_manifest.json"
with open(PATH_SCENARIO_B_MANIFEST, "w") as f:
    json.dump(manifest, f, indent=2)

print("Wrote manifest:", PATH_SCENARIO_B_MANIFEST)


Wrote manifest: data/results/scenario_B_manifest.json


### What Cell 6 Just Did

- Saved a lightweight provenance manifest capturing Scenario B’s slice rules, feature sources, and output paths.
- This makes Scenario B reproducible and easy to explain in the README / slides.


## Summary

Scenario B is now defined and reproducible.

- We created a deterministic Scenario B slice (top sponsors + status/phase gate).
- We engineered consistent optimization features (`_benefit_raw`, `_cost_raw`, `_safety_raw`).
- We exported:
  - `data/scenarios/scenario_B_trial_ids.csv`
  - `data/processed/scenario_B_candidates.csv`
  - (optional) `data/results/scenario_B_manifest.json`

Next: build `scenario_B_qubo.json` and the classical baseline in the Scenario B QUBO notebook.
